# E3.8 · Resilience over perfection

**Function E — Governance, Risk, Compliance & the CISO Office → The BISO, Risk Communicator & CISO Office**  ·  *Security of AI*

---

**Risk.** Trying to enumerate every failure mode of a probabilistic system.

**Control.** Maturity measured by containment, detection and recovery — not prevention.

**This lab.** Re-score your programme on containment/detection/recovery.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E3.8"))

Resilience over perfection. You will not prevent every agentic failure; the programme is judged on how quickly you notice, stop and recover.

In [ ]:
from cybercommons import ir, soc, grc
import time

now = time.time()
# notice
base = soc.Baseline({"read_file": 0.9, "http_get": 0.1}, actions_per_hour=300)
drift = base.compare([soc.Event(now, "a", "run_shell")] * 20 +
                     [soc.Event(now, "a", "read_file")] * 5)
print("notice:", drift["verdict"], "| new tools:", drift["new_tools"])

# stop
race = ir.containment_race(300, human_approval_minutes=8, auto_containment_seconds=12)
print(f"stop:   automated {race['actions_during_auto_containment']:.0f} further actions "
      f"vs {race['actions_during_manual_approval']:.0f} manual")

# recover
ok, missing = ir.Replay(["p"], ["r"], "glm-4.6@2025-11", 42).replayable()
print("recover: replayable =", ok, "| missing:", missing or "nothing")

Three capabilities, each independently testable, none of them prevention. A programme with all three survives a failure it did not predict — which is the only kind that actually happens.

In [ ]:
print(grc.SEQUENCING)
print("\nPerfection would mean step 3 (containment) never fails.")
print("Resilience means steps 4-6 work when it does.")

### Expect

Drift is detected with `run_shell` as a new tool, automated containment permits ~60 further actions against ~2400 for manual approval, and the instrumented run is replayable.

### Your turn

Run a game day that assumes containment failed. Measure notice, stop and recover as three separate numbers. The weakest one is next quarter's plan.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E3.8.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*